<a href="https://colab.research.google.com/github/peremartra/optipfair/blob/main/examples/knowledge_distillation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OptiPFair Notebook Series - Example: Knowledge Distillation\n
\n
![optiPfair Logo](https://github.com/peremartra/optipfair/blob/main/images/optiPfair.png?raw=true)\n
\n
This notebook demonstrates how to use [OptiPFair](https://github.com/peremartra/optipfair) to recover performance after depth pruning using knowledge distillation.  \n
It follows the full recommended workflow with public APIs only: load a teacher model, create a depth-pruned student, distill knowledge, and visualize training curves.\n
\n
The benchmark stage with lm_eval is intentionally excluded in this example.\n
\n
##Recommended Environment\n
\n
- **Platform**: [Google Colab](https://colab.research.google.com)  \n
- **Hardware**: GPU runtime (recommended: T4 or better for 0.8B-1B models)  \n
- **Dependencies**: Installed in Section 0\n
\n
##by Pere Martra.\n
\n
- [LinkedIn](https://www.linkedin.com/in/pere-martra)  \n
- [GitHub](https://github.com/peremartra)  \n
- [X / Twitter](https://x.com/peremartra)\n
\n
---\n
\n
> If you find this useful, please ⭐ the [repository](https://github.com/peremartra/optipfair) and share it!\n
\n
---\n
If you want your favorite LLM to create code with optiPfair, you just need to provide it with the file: [**optipfair_llm_reference_manual.txt**](https://github.com/peremartra/optipfair/blob/main/optipfair_llm_reference_manual.txt), which contains all the necessary information for the LLM to become an expert in using the library.

# Knowledge Distillation Example\n
\n
This notebook demonstrates how to recover a depth-pruned model with OptiPFair knowledge distillation.\n
We follow the recommended sequence for post-pruning recovery:\n
1. Load a teacher model\n
2. Prepare a small recovery dataset\n
3. Build a depth-pruned student with OptiPFair\n
4. Recover performance with knowledge distillation using `opf.distill_model()`\n
5. Plot training losses from `stats['loss_history']`

## 0. Environment and Dependencies\n
First, install the required libraries and define the base configuration.

In [ ]:
!pip install optipfair transformers datasets tqdm matplotlib

In [ ]:
RECOVERY_SAMPLES = 2000   # Use a small number for the example\n
EPOCHS = 3\n
LEARNING_RATE = 4e-5\n
BATCH_SIZE = 4\n
MAX_LENGTH = 512\n
LAYERS_TO_REMOVE_COUNT = 2

In [ ]:
import torch\n
from copy import deepcopy\n
from transformers import AutoModelForCausalLM, AutoTokenizer\n
from datasets import load_dataset, Dataset\n
from torch.utils.data import TensorDataset, DataLoader, random_split\n
from tqdm.auto import tqdm\n
import matplotlib.pyplot as plt\n
import optipfair as opf\n
\n
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n
print(f"Using device: {device}")

## 1. Load Teacher Model\n
Load the teacher model in evaluation mode and freeze all its parameters.\n
The teacher is only used as a supervision signal during distillation.

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B-Base"   # or "google/gemma-3-270m" for a smaller option\n
\n
print(f"Loading Teacher model: {MODEL_NAME}")\n
teacher_model = AutoModelForCausalLM.from_pretrained(\n
    MODEL_NAME,\n
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,\n
    device_map="auto" if torch.cuda.is_available() else None\n
)\n
\n
teacher_model.eval()\n
for param in teacher_model.parameters():\n
    param.requires_grad = False\n
\n
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)\n
if tokenizer.pad_token is None:\n
    tokenizer.pad_token = tokenizer.eos_token\n
\n
n_teacher_layers = teacher_model.config.num_hidden_layers\n
print(f"Teacher: {n_teacher_layers} layers, {teacher_model.num_parameters():,} params")

## 2. Prepare Training Dataset\n
Adapted from NB03: load Cosmopedia in streaming mode, tokenize text, and create an 80/20 train/validation split.

In [ ]:
print("Loading Cosmopedia dataset...")\n
dataset_name = "HuggingFaceTB/cosmopedia"\n
subsets = ["stories", "wikihow", "openstax", "web_samples_v1"]\n
samples_per_subset = int(RECOVERY_SAMPLES / len(subsets))\n
\n
all_samples = []\n
for subset in subsets:\n
    print(f"  Loading {subset}...")\n
    subset_data = load_dataset(dataset_name, subset, split="train", streaming=True)\n
    subset_samples = list(subset_data.take(samples_per_subset))\n
    all_samples.extend(subset_samples)\n
    print(f"    Collected {len(subset_samples):,} samples")\n
\n
distillation_dataset = Dataset.from_dict({"text": [s["text"] for s in all_samples]})\n
print(f"Total samples: {len(distillation_dataset):,}")

In [ ]:
print("Tokenizing...")\n
texts = [item["text"] for item in distillation_dataset]\n
tokenized_data = []\n
for i in tqdm(range(0, len(texts), 100), desc="Tokenizing"):\n
    batch = tokenizer(\n
        texts[i:i + 100],\n
        truncation=True,\n
        padding="max_length",\n
        max_length=MAX_LENGTH,\n
        return_tensors="pt"\n
    )\n
    tokenized_data.append(batch)\n
\n
input_ids = torch.cat([b["input_ids"] for b in tokenized_data], dim=0)\n
attention_mask = torch.cat([b["attention_mask"] for b in tokenized_data], dim=0)\n
full_dataset = TensorDataset(input_ids, attention_mask)\n
\n
generator = torch.Generator().manual_seed(42)\n
train_size = int(0.8 * len(full_dataset))\n
val_size = len(full_dataset) - train_size\n
train_dataset, val_dataset = random_split(\n
    full_dataset, [train_size, val_size], generator=generator\n
)\n
\n
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)\n
\n
print(f"Train: {len(train_dataset):,} samples ({len(train_dataloader):,} batches)")\n
print(f"Val:   {len(val_dataset):,} samples")

## 3. Create Pruned Student Model\n
Analyze layer importance with calibration data, remove the least important layers, and prepare the student for training.

In [ ]:
print("Analyzing layer importance...")\n
student_model = deepcopy(teacher_model)\n
importance_scores = opf.analyze_layer_importance(\n
    student_model,\n
    train_dataloader,\n
    show_progress=True\n
)\n
\n
print("\nLayer importance scores (lower = less important):")\n
for layer_idx, score in sorted(importance_scores.items()):\n
    print(f"  Layer {layer_idx:2d}: {score:.6f}")

In [ ]:
LAYERS_TO_REMOVE = sorted(\n
    importance_scores.keys(),\n
    key=lambda x: importance_scores[x]\n
)[:LAYERS_TO_REMOVE_COUNT]\n
\n
print(f"Layers selected for removal: {LAYERS_TO_REMOVE}")

In [ ]:
student_model = opf.prune_model_depth(\n
    model=student_model,\n
    layer_indices=LAYERS_TO_REMOVE,\n
    show_progress=True\n
)\n
\n
for param in student_model.parameters():\n
    param.requires_grad = True\n
\n
n_student_layers = student_model.config.num_hidden_layers\n
print(f"\nTeacher layers: {n_teacher_layers}")\n
print(f"Student layers: {n_student_layers} (removed {LAYERS_TO_REMOVE})")\n
print(f"Student params: {student_model.num_parameters():,}")

## 4. Knowledge Distillation with OptiPFair\n
Run labels-only distillation (hard labels + skew KLD) using the OptiPFair public API.\n
Feature alignment is disabled in this configuration (gamma=0, delta=0).

In [ ]:
student_to_train = deepcopy(student_model)\n
\n
trained_student, stats = opf.distill_model(\n
    student_model=student_to_train,\n
    teacher_model=teacher_model,\n
    dataloader=train_dataloader,\n
    alpha=0.6,\n
    beta=0.4,\n
    gamma=0.0,\n
    delta=0.0,\n
    temperature=2.0,\n
    skew_alpha=0.4,\n
    epochs=EPOCHS,\n
    learning_rate=LEARNING_RATE,\n
    accumulation_steps=4,\n
    show_progress=True,\n
    return_stats=True,\n
)\n
\n
print("\nTraining complete")\n
print(f"  Total time:        {stats['total_time_seconds']:.1f}s ({stats['total_time_seconds'] / 60:.1f} min)")\n
print(f"  Avg time/epoch:    {stats['avg_time_per_epoch']:.1f}s")\n
print(f"  Final total loss:  {stats['loss_history']['total'][-1]:.4f}")\n
print(f"  Final task loss:   {stats['loss_history']['task'][-1]:.4f}")\n
print(f"  Final logits loss: {stats['loss_history']['logits'][-1]:.4f}")

## 5. Visualize Training Loss Curves\n
Plot total, task, and logits losses using the stats dictionary returned by distill_model.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))\n
\n
loss_keys = [\n
    ("total",  "Total Loss"),\n
    ("task",   "Task Loss (Cross-Entropy)"),\n
    ("logits", "Logits Loss (Skew KLD)"),\n
]\n
\n
for idx, (key, title) in enumerate(loss_keys):\n
    axes[idx].plot(\n
        stats['loss_history'][key],\n
        marker='o', linewidth=2, markersize=8\n
    )\n
    axes[idx].set_title(title, fontsize=12)\n
    axes[idx].set_xlabel('Epoch')\n
    axes[idx].set_ylabel('Loss')\n
    axes[idx].grid(True, alpha=0.3)\n
\n
plt.suptitle(\n
    'Training Progress: Knowledge Distillation (Labels Only)',\n
    fontsize=14, fontweight='bold'\n
)\n
plt.tight_layout()\n
plt.savefig('kd_training_curves.png', dpi=150, bbox_inches='tight')\n
plt.show()\n
\n
print("Plot saved to kd_training_curves.png")

## 6. Optional Save Trained Student\n
Save the distilled student and tokenizer for later use.

In [ ]:
OUTPUT_PATH = "./kd-student"\n
trained_student.save_pretrained(OUTPUT_PATH)\n
tokenizer.save_pretrained(OUTPUT_PATH)\n
print(f"Trained student saved to {OUTPUT_PATH}")